In [1]:
import pandas as pd

In [2]:
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv()

class PostgresSQL():
    def __init__(self):
        self.host = "localhost"
        self.port = 5432
        self.database = "data_source"
        self.user = os.getenv("POSTGRES_USER")
        self.password = os.getenv("POSTGRES_PASSWORD")
 
    def connect(self):
        connection = psycopg2.connect(
            host=self.host,
            port=self.port,
            database=self.database,
            user=self.user,
            password=self.password
        )
        return connection
    
    def cursor(self):
        return self.connect().cursor()
    
    def close(self):
        self.connect().close()


In [3]:
p = PostgresSQL()
conn = p.connect()
c = conn.cursor()

In [5]:
c.execute("SELECT COUNT(*) FROM transactions")
pd.DataFrame(c.fetchall(), columns=[desc[0] for desc in c.description])

,count
0,79


In [5]:
c.execute("SELECT COUNT(*) FROM customers")
c.fetchone()

(185,)

In [ ]:
p.close()

In [1]:
test = []
test.append(1)
test.pop()

1

In [3]:
i = 2
for i in range(5):
    print(i)

0
1
2
3
4


### Test code clickhouse

In [8]:
import clickhouse_connect
client = clickhouse_connect.get_client(host='localhost', port=8123, user='clickhouse', password='clickhouse')
print(client.command("SHOW DATABASES"))

INFORMATION_SCHEMA
default
information_schema
streaming
system


In [9]:
client.command("""
CREATE TABLE IF NOT EXISTS streaming.kafka_enriched_transactions
(
    transaction_id     String,
    customer_id        Int32,
    customer_name      String,
    customer_city      String,
    merchant_id        Int32,
    merchant_name      String,
    merchant_category  String,
    amount             Decimal(15, 2),
    currency           String,
    status             String,
    payment_method     String,
    transaction_time   DateTime64(3)
)
ENGINE = Kafka()
SETTINGS
    kafka_broker_list      = 'kafka:9092',
    kafka_topic_list       = 'processed.enriched_transactions',
    kafka_group_name       = 'clickhouse-enriched-transactions',
    kafka_format           = 'AvroConfluent',
    format_avro_schema_registry_url = 'http://schema-registry:8081',
    kafka_handle_error_mode = 'stream';
""")

In [12]:
client.command("""
CREATE TABLE IF NOT EXISTS streaming.enriched_transactions
(
    transaction_id     String,
    customer_id        Int32,
    customer_name      String,
    customer_city      String,
    merchant_id        Int32,
    merchant_name      String,
    merchant_category  String,
    amount             Decimal(15, 2),
    currency           String,
    status             String,
    payment_method     String,
    transaction_time   DateTime64(3),
    ingested_at        DateTime DEFAULT now()
)
ENGINE = ReplacingMergeTree(transaction_time) -- If have any duplicate transaction_id, keep the one with the latest transaction_time
PARTITION BY toYYYYMM(transaction_time)
ORDER BY transaction_id;

""")

In [13]:
client.command("""
CREATE MATERIALIZED VIEW IF NOT EXISTS streaming.mv_enriched_transactions 
TO streaming.enriched_transactions AS
SELECT
    transaction_id,
    customer_id,
    customer_name,
    customer_city,
    merchant_id,
    merchant_name,
    merchant_category,
    amount,
    currency,
    status,
    payment_method,
    transaction_time
FROM streaming.kafka_enriched_transactions;
""")

In [15]:
print(client.command("SHOW TABLES FROM streaming"))

enriched_transactions
kafka_enriched_transactions
mv_enriched_transactions


In [17]:
client.query_df("SELECT * FROM streaming.enriched_transactions LIMIT 10")

,transaction_id,customer_id,customer_name,customer_city,merchant_id,merchant_name,merchant_category,amount,currency,status,payment_method,transaction_time,ingested_at
0,,0,,,0,,,0.00,,,,1970-01-01,2026-05-02 10:28:26
